In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
"""
==============================================================================
NOTEBOOK 1: DATA EXPLORATION & STRATIFIED SAMPLING
==============================================================================

Project: Industrial-Scale Uplift Modeling for Ad Spend Optimization
Dataset: Criteo Uplift Modeling v2.1 (13.9M users)
Goal:    Create memory-efficient stratified sample preserving treatment-
         outcome distributions for causal inference experiments.

Author:  Anurag Jain (M.Sc. Economics, IGIDR)
Date:    31/05/2026
==============================================================================
"""

# Imports
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Set styling
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Environment setup complete")

In [ ]:
import os

# Check what's in the input directory
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        path = os.path.join(dirname, filename)
        size_gb = os.path.getsize(path) / (1024**3)
        print(f"{path} | Size: {size_gb:.2f} GB")

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm

# Path to dataset
DATA_PATH = '/kaggle/input/datasets/evgeniypolin/criteo-uplift-v2-1/criteo-uplift-v2.1.csv'

# First, just peek at the structure (load first 1000 rows)
df_peek = pd.read_csv(DATA_PATH, nrows=1000)
print("Columns:", df_peek.columns.tolist())
print("\nDataset preview:")
df_peek.head()
print("\nData types:")
print(df_peek.dtypes)

In [ ]:
# Count total rows (without loading)
total_rows = sum(1 for _ in open(DATA_PATH)) - 1  # -1 for header
print(f"Total rows: {total_rows:,}")

In [ ]:
# Load in chunks and sample stratified
chunk_size = 1_000_000  # 1M rows at a time
sample_fraction = 0.15  # ~2M row sample

print("📥 Loading data in chunks and sampling...")
sampled_chunks = []

chunks = pd.read_csv(DATA_PATH, chunksize=chunk_size)

for i, chunk in enumerate(tqdm(chunks)):
    # Stratified sample preserving treatment + conversion distribution
    sampled = (
        chunk.groupby(['treatment', 'conversion'], group_keys=False)
        .apply(lambda x: x.sample(frac=sample_fraction, random_state=42))
    )
    sampled_chunks.append(sampled)
    
    # Optional: print progress
    if i % 5 == 0:
        print(f"Chunk {i}: {len(sampled):,} rows sampled")

# Combine all sampled chunks
df_sample = pd.concat(sampled_chunks, ignore_index=True)
print(f"\n✅ Final sample size: {len(df_sample):,} rows")
print(f"Memory usage: {df_sample.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
def optimize_memory(df):
    """Downcast numerical columns to save memory"""
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')
    for col in df.select_dtypes(include=['int64']).columns:
        df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

df_sample = optimize_memory(df_sample)
print(f"Optimized memory: {df_sample.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
# Check that we preserved treatment & outcome distributions
print("📊 Treatment distribution:")
print(df_sample['treatment'].value_counts(normalize=True))

print("\n📊 Conversion rate by treatment:")
print(df_sample.groupby('treatment')['conversion'].mean())

print("\n📊 Visit rate by treatment:")
print(df_sample.groupby('treatment')['visit'].mean())

# Naive ATE (just for reference — we'll do proper causal analysis later)
ate_naive = (
    df_sample[df_sample['treatment']==1]['conversion'].mean() - 
    df_sample[df_sample['treatment']==0]['conversion'].mean()
)
print(f"\n💡 Naive ATE: {ate_naive:.5f} ({ate_naive*100:.3f}% lift)")

In [ ]:
# Save to Kaggle working directory
OUTPUT_PATH = '/kaggle/working/criteo_sample_2M.parquet'
df_sample.to_parquet(OUTPUT_PATH, compression='snappy', index=False)

import os
file_size = os.path.getsize(OUTPUT_PATH) / (1024**2)
print(f"✅ Saved Parquet file: {OUTPUT_PATH}")
print(f"   File size: {file_size:.2f}")

In [ ]:
# Test loading back from Parquet
df_test = pd.read_parquet(OUTPUT_PATH)
print(f"Loaded {len(df_test):,} rows successfully")
print(f"Columns: {df_test.columns.tolist()}")
df_test.head()